In [1]:
import os
import json
import cv2
import shutil
import random
import copy
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
import albumentations as A
from tqdm import tqdm

# =====================================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN & THAM SỐ (GIỮ NGUYÊN GỐC)
# =====================================================================
INPUT_DATASETS = [
    {
        "prefix": "gold",
        "img_dir": Path("/kaggle/input/datasets/quii29/rukopys-dataset/train/images"),
        "meta_path": Path("/kaggle/input/datasets/quii29/rukopys-dataset/train/metadata.jsonl")
    },
    {
        "prefix": "silver",
        "img_dir": Path("/kaggle/input/datasets/quii29/rukopys-silver-rare-classes/images"),
        "meta_path": Path("/kaggle/input/datasets/quii29/rukopys-silver-rare-classes/metadata.jsonl")
    }
]

WORKING_DIR = Path("/kaggle/working")

OUT_TRAIN_ROOT = WORKING_DIR / "Merged_Rukopys_V1"
OUT_TRAIN_IMG = OUT_TRAIN_ROOT / "train" / "images"
OUT_TRAIN_META = OUT_TRAIN_ROOT / "train" / "metadata.jsonl"
OUT_TRAIN_COCO = OUT_TRAIN_ROOT / "train" / "annotations_detection.json"
OUT_TRAIN_REL  = OUT_TRAIN_ROOT / "train" / "annotations_relations.json"

OUT_TEST_ROOT = WORKING_DIR / "TestFromTrain"
OUT_TEST_IMG  = OUT_TEST_ROOT / "images"
OUT_TEST_META = OUT_TEST_ROOT / "metadata.jsonl"
OUT_TEST_COCO = OUT_TEST_ROOT / "annotations_detection.json"
OUT_TEST_REL  = OUT_TEST_ROOT / "annotations_relations.json"

DEBUG_DIR = WORKING_DIR / "debug_merged"
REPORT_IMG_PATH = WORKING_DIR / "class_distribution_report.png"

BBOX_FORMAT = "coco" 
MAX_REPEATS = 4
MAX_IMAGE_SIZE = 4000 
TEST_RATIO = 0.20

HEAD_CLASSES = {'handwritten', 'formula'}
MID_CLASSES = {'printed', 'annotation', 'table'}
TAIL_CLASSES = {'image', 'graph'}

READABLE_CLASSES = {'handwritten', 'printed', 'annotation', 'formula'}
CLASS_NAME_TO_ID = {
    'handwritten': 1, 'printed': 2, 'formula': 3, 
    'table': 4, 'annotation': 5, 'image': 6, 'graph': 7
}

# =====================================================================
# 2. HÀM CHUYỂN ĐỔI BBOX & EDGE (KHÔI PHỤC NGUYÊN BẢN 100%)
# =====================================================================
def original_bbox_to_coco(bbox, fmt):
    if fmt == "coco": return bbox
    if fmt == "xyxy": return [bbox[0], bbox[1], bbox[2] - bbox[0], bbox[3] - bbox[1]]
    raise ValueError(f"Unsupported format: {fmt}")

def coco_bbox_to_original(bbox, fmt):
    if fmt == "coco": return bbox
    if fmt == "xyxy": return [bbox[0], bbox[1], bbox[0] + bbox[2], bbox[1] + bbox[3]]
    raise ValueError(f"Unsupported format: {fmt}")

def clamp_coco_bbox(bbox, img_w, img_h):
    x, y, w, h = bbox
    x = max(0.0, min(float(x), float(img_w) - 2.0))
    y = max(0.0, min(float(y), float(img_h) - 2.0))
    w = max(1.0, min(float(w), float(img_w) - x))
    h = max(1.0, min(float(h), float(img_h) - y))
    return [x, y, w, h]

def generate_reading_order_edges(regions):
    readable_items = []
    for idx, r in enumerate(regions):
        c_type = r.get('type', '')
        if c_type in READABLE_CLASSES and 'bbox' in r:
            x, y, w, h = r['bbox']
            if BBOX_FORMAT != "coco":
                x, y, w, h = original_bbox_to_coco([x,y,w,h], BBOX_FORMAT)
            cx = x + w / 2.0
            cy = y + h / 2.0
            readable_items.append({'idx': idx, 'cx': cx, 'cy': cy, 'h': h})
            
    if not readable_items: return []
        
    readable_items.sort(key=lambda item: item['cy'])
    
    lines = []
    current_line = [readable_items[0]]
    for item in readable_items[1:]:
        prev_item = current_line[-1]
        threshold = max(prev_item['h'], item['h']) * 0.5 
        if abs(item['cy'] - prev_item['cy']) < threshold:
            current_line.append(item)
        else:
            lines.append(current_line)
            current_line = [item]
    if current_line: lines.append(current_line)
        
    ordered_indices = []
    for line in lines:
        line.sort(key=lambda item: item['cx'])
        ordered_indices.extend([item['idx'] for item in line])
        
    edges = []
    for i in range(len(ordered_indices) - 1):
        edges.append([ordered_indices[i], ordered_indices[i+1], "next"])
        
    return edges

# =====================================================================
# 3. LOGIC TÁI CÂN BẰNG (KHÔI PHỤC NGUYÊN BẢN 100%)
# =====================================================================
def compute_image_priority(regions):
    counts = defaultdict(int)
    for r in regions: counts[r.get('type', 'unknown')] += 1
        
    total_boxes = sum(counts.values())
    if total_boxes == 0: return {"repeats": 0, "pipeline": "aug_light", "stats": counts}

    head_boxes = sum(counts[c] for c in HEAD_CLASSES)
    mid_boxes = sum(counts[c] for c in MID_CLASSES)
    tail_boxes = sum(counts[c] for c in TAIL_CLASSES)
    
    rare_ratio = tail_boxes / total_boxes
    dominant_ratio = head_boxes / total_boxes

    if dominant_ratio >= 0.8 and tail_boxes <= 1:
        return {"repeats": 0, "pipeline": "aug_light", "stats": counts}

    score = (tail_boxes * 3.0 + mid_boxes * 1.2 + rare_ratio * 8.0 + counts['image'] * 2.0 + counts['graph'] * 3.0) - (dominant_ratio * 6.0 + max(0, counts['handwritten'] - 20) * 0.15 + max(0, counts['formula'] - 10) * 0.20)

    repeats = 0
    pipeline = "aug_light"

    if tail_boxes > 0 and rare_ratio >= 0.1:
        repeats = random.choice([3, 4])
        pipeline = "aug_rare"
    elif tail_boxes > 0 or mid_boxes > 5:
        repeats = random.choice([1, 2])
        pipeline = "aug_rare" if tail_boxes > 0 else "aug_dense"
    elif counts['handwritten'] >= 15:
        repeats = 1
        pipeline = "aug_dense"
    elif score > 0:
        repeats = random.choice([0, 1])
        pipeline = "aug_light"

    return {"repeats": min(repeats, MAX_REPEATS), "pipeline": pipeline, "stats": counts}

# =====================================================================
# 4. THUẬT TOÁN STRATIFIED SPLIT & HELPER
# =====================================================================
def greedy_rarity_stratified_split(items, ratio=0.2, seed=42):
    random.seed(seed)
    class_counts = defaultdict(int)
    for it in items:
        for r in it['record'].get('regions', []):
            if r.get('type') in CLASS_NAME_TO_ID:
                class_counts[r.get('type')] += 1
                
    sorted_classes = sorted(CLASS_NAME_TO_ID.keys(), key=lambda c: class_counts[c])
    test_items, train_items = [], []
    assigned_ids = set()
    target_test_len = int(round(len(items) * ratio))
    
    for cls in sorted_classes:
        candidates = [i for i, it in enumerate(items) 
                      if i not in assigned_ids and any(r.get('type') == cls for r in it['record'].get('regions', []))]
        random.shuffle(candidates)
        take = int(round(len(candidates) * ratio))
        
        for idx in candidates[:take]:
            test_items.append(items[idx])
            assigned_ids.add(idx)
        for idx in candidates[take:]:
            train_items.append(items[idx])
            assigned_ids.add(idx)
            
    unassigned = [i for i in range(len(items)) if i not in assigned_ids]
    random.shuffle(unassigned)
    needed = max(0, target_test_len - len(test_items))
    
    test_items.extend([items[i] for i in unassigned[:needed]])
    train_items.extend([items[i] for i in unassigned[needed:]])
    return train_items, test_items

def create_coco_skeleton():
    return {"images": [], "annotations": [], "categories": [{"id": v, "name": k} for k, v in CLASS_NAME_TO_ID.items()]}

# =====================================================================
# 5. AUGMENTATION PIPELINES (GIỮ NGUYÊN GỐC)
# =====================================================================
bbox_params = A.BboxParams(format='coco', label_fields=['region_idx'], min_visibility=0.5)

pipelines = {
    "aug_light": A.Compose([A.RandomBrightnessContrast(p=0.5), A.GaussNoise(p=0.3), A.Blur(blur_limit=3, p=0.2)], bbox_params=bbox_params),
    "aug_dense": A.Compose([A.Affine(rotate=(-3, 3), translate_percent={"x": (-0.02, 0.02), "y": (-0.02, 0.02)}, p=0.7), A.RandomBrightnessContrast(p=0.5), A.ISONoise(p=0.3)], bbox_params=bbox_params),
    "aug_rare":  A.Compose([A.Affine(rotate=(-3, 3), scale=(0.97, 1.03), p=0.8), A.RandomBrightnessContrast(p=0.6)], bbox_params=bbox_params)
}

# =====================================================================
# 6. CHƯƠNG TRÌNH CHÍNH
# =====================================================================
def main():
    for p in [OUT_TRAIN_ROOT, OUT_TEST_ROOT, DEBUG_DIR]:
        if p.exists(): shutil.rmtree(p)
    OUT_TRAIN_IMG.mkdir(parents=True, exist_ok=True)
    OUT_TEST_IMG.mkdir(parents=True, exist_ok=True)
    DEBUG_DIR.mkdir(parents=True, exist_ok=True)

    print("🚀 BƯỚC 1: Đọc và phân tích Metadata từ các Nguồn...")
    gold_items, silver_items = [], []
    
    for ds in INPUT_DATASETS:
        if not ds["meta_path"].exists(): continue
        with open(ds["meta_path"], 'r', encoding='utf-8') as f:
            for line in f:
                rec = json.loads(line)
                item = {"prefix": ds["prefix"], "img_dir": ds["img_dir"], "record": rec}
                if ds["prefix"] == "gold": gold_items.append(item)
                else: silver_items.append(item)

    gold_train, gold_test = greedy_rarity_stratified_split(gold_items, ratio=TEST_RATIO)
    train_pool = gold_train + silver_items
    test_pool  = gold_test
    
    print(f"✂️ Đã chia tập Gold: {len(gold_train)} Train | {len(gold_test)} Test (TestFromTrain)")

    report_stats = {
        "gold_full": defaultdict(int), "test": defaultdict(int),
        "train_pre": defaultdict(int), "train_fin": defaultdict(int)
    }

    def process_image_file(item, out_img_dir):
        prefix, rec_orig = item["prefix"], item["record"]
        rec = copy.deepcopy(rec_orig)
        orig_fname = Path(rec.get('file_name', '')).name
        safe_fname = f"{prefix}_{orig_fname}"
        in_path = item["img_dir"] / orig_fname
        
        if not in_path.exists(): return None, None, None
        img = cv2.imread(str(in_path))
        if img is None: return None, None, None
        
        img_h, img_w = img.shape[:2]
        regions = rec.get('regions', [])
        
        if img_w > MAX_IMAGE_SIZE or img_h > MAX_IMAGE_SIZE:
            scale = MAX_IMAGE_SIZE / max(img_w, img_h)
            new_w, new_h = int(img_w * scale), int(img_h * scale)
            img = cv2.resize(img, (new_w, new_h))
            img_w, img_h = new_w, new_h
            for r in regions:
                if 'bbox' in r and len(r['bbox']) == 4:
                    box = original_bbox_to_coco(r['bbox'], BBOX_FORMAT)
                    box = [box[0]*scale, box[1]*scale, box[2]*scale, box[3]*scale]
                    r['bbox'] = coco_bbox_to_original(box, BBOX_FORMAT)

        out_path = out_img_dir / safe_fname
        if not out_path.exists(): cv2.imwrite(str(out_path), img)
        rec['image_width'], rec['image_height'] = img_w, img_h
        rec['file_name'] = f"images/{safe_fname}"
        return img, rec, safe_fname

    # ==========================================
    # PHASE A: XỬ LÝ TẬP TEST
    # ==========================================
    print("\n📦 Xuất tập TestFromTrain...")
    test_coco, test_rel = create_coco_skeleton(), []
    t_img_id, t_ann_id = 1, 1
    
    with open(OUT_TEST_META, 'w', encoding='utf-8') as f_meta:
        for item in tqdm(test_pool, desc="Building Test"):
            img, rec, safe_name = process_image_file(item, OUT_TEST_IMG)
            if img is None: continue
            
            f_meta.write(json.dumps(rec, ensure_ascii=False) + '\n')
            test_coco["images"].append({"id": t_img_id, "file_name": rec['file_name'], "width": rec['image_width'], "height": rec['image_height']})
            rel_rec = {"image_id": t_img_id, "file_name": rec['file_name'], "regions": [], "edges": generate_reading_order_edges(rec['regions'])}
            
            for r_idx, r in enumerate(rec.get('regions', [])):
                c_type = r.get('type')
                report_stats["test"][c_type] += 1
                report_stats["gold_full"][c_type] += 1
                rel_rec["regions"].append({"region_idx": r_idx, "type": c_type, "bbox": r.get('bbox')})
                
                cat_id = CLASS_NAME_TO_ID.get(c_type)
                if cat_id and 'bbox' in r:
                    box = original_bbox_to_coco(r['bbox'], BBOX_FORMAT)
                    test_coco["annotations"].append({"id": t_ann_id, "image_id": t_img_id, "category_id": cat_id, "bbox": [round(float(v),2) for v in box], "area": round(float(box[2]*box[3]),2), "iscrowd": 0})
                    t_ann_id += 1
            test_rel.append(rel_rec)
            t_img_id += 1

    with open(OUT_TEST_COCO, 'w', encoding='utf-8') as f: json.dump(test_coco, f, ensure_ascii=False, indent=2)
    with open(OUT_TEST_REL, 'w', encoding='utf-8') as f: json.dump(test_rel, f, ensure_ascii=False, indent=2)

    # ==========================================
    # PHASE B: XỬ LÝ TRAIN & AUGMENT
    # ==========================================
    print("\n🔥 Xuất tập Train & Bơm nhiễu...")
    train_coco, train_rel = create_coco_skeleton(), []
    tr_img_id, tr_ann_id = 1, 1
    debug_samples = []
    
    with open(OUT_TRAIN_META, 'w', encoding='utf-8') as f_meta:
        for item in tqdm(train_pool, desc="Building Train"):
            img, rec, safe_name = process_image_file(item, OUT_TRAIN_IMG)
            if img is None: continue
            
            f_meta.write(json.dumps(rec, ensure_ascii=False) + '\n')
            img_w, img_h = rec['image_width'], rec['image_height']
            train_coco["images"].append({"id": tr_img_id, "file_name": rec['file_name'], "width": img_w, "height": img_h})
            rel_rec = {"image_id": tr_img_id, "file_name": rec['file_name'], "regions": [], "edges": generate_reading_order_edges(rec['regions'])}
            
            valid_bboxes, valid_indices = [], []
            for r_idx, r in enumerate(rec.get('regions', [])):
                c_type = r.get('type')
                if item["prefix"] == "gold": report_stats["gold_full"][c_type] += 1
                report_stats["train_pre"][c_type] += 1
                report_stats["train_fin"][c_type] += 1
                rel_rec["regions"].append({"region_idx": r_idx, "type": c_type, "bbox": r.get('bbox')})
                
                cat_id = CLASS_NAME_TO_ID.get(c_type)
                if cat_id and 'bbox' in r:
                    box = clamp_coco_bbox(original_bbox_to_coco(r['bbox'], BBOX_FORMAT), img_w, img_h)
                    if box[2] > 1 and box[3] > 1:
                        valid_bboxes.append(box)
                        valid_indices.append(r_idx)
                        train_coco["annotations"].append({"id": tr_ann_id, "image_id": tr_img_id, "category_id": cat_id, "bbox": [round(float(v),2) for v in box], "area": round(float(box[2]*box[3]),2), "iscrowd": 0})
                        tr_ann_id += 1
                        
            train_rel.append(rel_rec)
            tr_img_id += 1
            
            if not valid_bboxes: continue
            prio = compute_image_priority(rec['regions'])
            if prio["repeats"] == 0: continue
            
            transform = pipelines[prio["pipeline"]]
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            for i in range(prio["repeats"]):
                try: aug = transform(image=img_rgb, bboxes=valid_bboxes, region_idx=valid_indices)
                except Exception: continue
                if not aug['bboxes']: continue
                
                new_fname = f"{Path(safe_name).stem}_{prio['pipeline']}_aug{i+1:03d}{Path(safe_name).suffix}"
                cv2.imwrite(str(OUT_TRAIN_IMG / new_fname), cv2.cvtColor(aug['image'], cv2.COLOR_RGB2BGR))
                
                new_rec = copy.deepcopy(rec)
                new_rec['file_name'], new_rec['image_width'], new_rec['image_height'] = f"images/{new_fname}", aug['image'].shape[1], aug['image'].shape[0]
                new_regions = []
                
                for n_box, o_idx in zip(aug['bboxes'], aug['region_idx']):
                    rg = copy.deepcopy(rec['regions'][int(o_idx)])
                    final_b = clamp_coco_bbox(coco_bbox_to_original(n_box, BBOX_FORMAT), new_rec['image_width'], new_rec['image_height'])
                    rg['bbox'] = [round(float(v),2) for v in final_b]
                    new_regions.append(rg)
                    report_stats["train_fin"][rg.get('type')] += 1
                    
                new_rec['regions'] = new_regions
                f_meta.write(json.dumps(new_rec, ensure_ascii=False) + '\n')
                
                train_coco["images"].append({"id": tr_img_id, "file_name": new_rec['file_name'], "width": new_rec['image_width'], "height": new_rec['image_height']})
                for r_idx, r in enumerate(new_regions):
                    cat_id = CLASS_NAME_TO_ID.get(r.get('type'))
                    if cat_id:
                        b = original_bbox_to_coco(r['bbox'], BBOX_FORMAT)
                        train_coco["annotations"].append({"id": tr_ann_id, "image_id": tr_img_id, "category_id": cat_id, "bbox": [round(float(v),2) for v in b], "area": round(float(b[2]*b[3]),2), "iscrowd": 0})
                        tr_ann_id += 1
                tr_img_id += 1
                
                # KHÔI PHỤC LẠI LOGIC SAMPLE DEBUG GỐC CỦA BẠN
                if prio["pipeline"] == "aug_rare" or (random.random() < 0.1 and len(debug_samples) < 30):
                    if len(debug_samples) < 30: debug_samples.append((OUT_TRAIN_IMG / new_fname, new_rec))

    with open(OUT_TRAIN_COCO, 'w', encoding='utf-8') as f: json.dump(train_coco, f, ensure_ascii=False, indent=2)
    with open(OUT_TRAIN_REL, 'w', encoding='utf-8') as f: json.dump(train_rel, f, ensure_ascii=False, indent=2)

    # KHÔI PHỤC ĐOẠN VẼ DEBUG VISUALIZATION GỐC
    print("\n🔍 Đang tạo Debug Visualization...")
    for img_p, rec in debug_samples:
        dbg_img = cv2.imread(str(img_p))
        if dbg_img is None: continue
        for r in rec.get('regions', []):
            box = r['bbox']
            if BBOX_FORMAT == "coco": x1, y1, x2, y2 = int(box[0]), int(box[1]), int(box[0]+box[2]), int(box[1]+box[3])
            else: x1, y1, x2, y2 = int(box[0]), int(box[1]), int(box[2]), int(box[3])
            cv2.rectangle(dbg_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(dbg_img, r.get('type',''), (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
        cv2.imwrite(str(DEBUG_DIR / img_p.name), dbg_img)

    # ==========================================
    # PHASE C: THỐNG KÊ & NÉN ZIP
    # ==========================================
    print("\n" + "="*65)
    print("📊 BẢNG SỐ LIỆU ĐỐI CHIẾU CHÍNH THỨC (COPY VÀO LATEX)")
    print("="*65)
    classes = sorted(CLASS_NAME_TO_ID.keys())
    print(f"| Class | Gold Gốc | TestFromTrain (~20%) | Train Trước Aug | Train Sau Aug | Boost |")
    print(f"| :--- | :---: | :---: | :---: | :---: | :---: |")
    for c in classes:
        gf, tb = report_stats["gold_full"][c], report_stats["test"][c]
        tp, tf = report_stats["train_pre"][c], report_stats["train_fin"][c]
        rat = f"{(tb/gf)*100:.1f}%" if gf > 0 else "0%"
        bst = f"{tf/tp:.2f}x" if tp > 0 else "N/A"
        print(f"| `{c}` | {gf} | {tb} ({rat}) | {tp} | **{tf}** | {bst} |")

    x = np.arange(len(classes))
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - 0.25, [report_stats["gold_full"][c] for c in classes], 0.25, label='Gold Gốc', color='#95a5a6')
    ax.bar(x,        [report_stats["test"][c] for c in classes],      0.25, label='TestFromTrain (20%)', color='#e74c3c')
    ax.bar(x + 0.25, [report_stats["train_fin"][c] for c in classes], 0.25, label='Train Sau Augment', color='#2ecc71')
    ax.set_title('Phân phối Bounding Box các class'); ax.set_xticks(x); ax.set_xticklabels(classes)
    ax.legend(); plt.tight_layout(); plt.savefig(REPORT_IMG_PATH, dpi=300); plt.close()

    print("\n🗜️ Đang nén Zip...")
    shutil.make_archive(str(WORKING_DIR / "TestFromTrain"), 'zip', OUT_TEST_ROOT)
    shutil.make_archive(str(WORKING_DIR / "Merged_Rukopys_V1"), 'zip', OUT_TRAIN_ROOT)

    shutil.rmtree(OUT_TEST_ROOT); shutil.rmtree(OUT_TRAIN_ROOT)
    print("✅ Xong! Output lưu giữ: TestFromTrain.zip, Merged_Rukopys_V1.zip, debug_merged/ và ảnh biểu đồ.")

if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()


🚀 BƯỚC 1: Đọc và phân tích Metadata từ các Nguồn...
✂️ Đã chia tập Gold: 1064 Train | 266 Test (TestFromTrain)

📦 Xuất tập TestFromTrain...


Building Test: 100%|██████████| 266/266 [00:23<00:00, 11.19it/s]



🔥 Xuất tập Train & Bơm nhiễu...


Building Train: 100%|██████████| 1743/1743 [03:14<00:00,  8.94it/s]



🔍 Đang tạo Debug Visualization...

📊 BẢNG SỐ LIỆU ĐỐI CHIẾU CHÍNH THỨC (COPY VÀO LATEX)
| Class | Gold Gốc | TestFromTrain (~20%) | Train Trước Aug | Train Sau Aug | Boost |
| :--- | :---: | :---: | :---: | :---: | :---: |
| `annotation` | 554 | 101 (18.2%) | 956 | **1746** | 1.83x |
| `formula` | 2892 | 615 (21.3%) | 3425 | **4200** | 1.23x |
| `graph` | 59 | 10 (16.9%) | 88 | **247** | 2.81x |
| `handwritten` | 21577 | 4327 (20.1%) | 23324 | **28800** | 1.23x |
| `image` | 119 | 21 (17.6%) | 535 | **1547** | 2.89x |
| `printed` | 308 | 21 (6.8%) | 3689 | **9773** | 2.65x |
| `table` | 142 | 38 (26.8%) | 649 | **1013** | 1.56x |

🗜️ Đang nén Zip...
✅ Xong! Output lưu giữ: TestFromTrain.zip, Merged_Rukopys_V1.zip, debug_merged/ và ảnh biểu đồ.
